In [1]:
MODE = 'PDE'

import pandas as pd
import numpy as np
import os
import glob
from tqdm import tqdm

# Data loading based on the MODE
if MODE == 'PDE':
    df = pd.read_csv('output/pde/catalogue/events_filtered.csv')
    protagonists_df = pd.read_csv('output/individual/catalogue/protagonists_master.csv')

# Home team extraction from match files
files = glob.glob("sb_offline_data/match_*.pkl")
home_teams_dict = {}

for file in files:
    m_id = int(os.path.basename(file).split('_')[1].split('.')[0])
    events = pd.read_pickle(file)
    starting_xis = events[events['type'] == 'Starting XI']
    if not starting_xis.empty:
        home_teams_dict[m_id] = starting_xis.iloc[0]['team']
    else:
        home_teams_dict[m_id] = "Unknown"

In [2]:
def get_player_metrics(player_df, target_player_id, window_sec=300, event_type=None, is_post_window=False):
        
    res = {m: 0.0 for m in ['pass_acc_ratio', 'fwd_pass_ratio', 'long_ball_ratio', 'xg_rate', 
                            'shot_rate', 'avg_xg', 'dribble_rate', 'turnover_rate', 
                            'foul_rate', 'press_intensity']}
    
    if player_df.empty or window_sec <= 0:
        return res

    # Filter for the specific player (Biztonsági szűrés visszatéve)
    # player_df = player_df[player_df['player_id'] == target_player_id]
        
    # Filter out non-open play events for passing and shooting metrics
    if 'pass_type' in player_df.columns:
        player_df = player_df[~player_df['pass_type'].isin(['Corner', 'Free Kick', 'Goal Kick', 'Throw-in'])]
    if 'shot_type' in player_df.columns:
        player_df = player_df[~player_df['shot_type'].isin(['Penalty', 'Free Kick'])]

    type_counts = player_df['type'].value_counts().to_dict()
    
    # Passing Metrics
    n_passes = type_counts.get('Pass', 0)
    if n_passes > 0:
        passes = player_df[player_df['type'] == 'Pass']
        # Pass completion rate: Successful / Total
        res['pass_acc_ratio'] = passes['pass_outcome'].isnull().sum() / n_passes
        # Forward pass ratio: Forward / Total
        res['fwd_pass_ratio'] = (passes['end_x'] > passes['x']).sum() / n_passes
        # Long ball ratio: Long (>32m) / Total
        x_diff = passes['end_x'].values - passes['x'].values
        y_diff = passes['end_y'].values - passes['y'].values
        res['long_ball_ratio'] = (np.sqrt(x_diff**2 + y_diff**2) > 32).sum() / n_passes
    else:
        res['pass_acc_ratio'] = res['fwd_pass_ratio'] = res['long_ball_ratio'] = 0.0

    # Shooting Metrics
    n_shots = type_counts.get('Shot', 0)
    if n_shots > 0:
        shots = player_df[player_df['type'] == 'Shot']
        if 'total_seconds' in shots.columns:
            shots = shots.sort_values('total_seconds')

        if is_post_window and event_type in {'Converted Turnover', 'Final Third Dribble'} and len(shots) > 0:
            shots = shots.iloc[1:]
            
        n_shots = len(shots)
        res['xg_rate'] = shots['shot_statsbomb_xg'].sum() if n_shots > 0 else 0.0
        res['shot_rate'] = n_shots
        res['avg_xg'] = shots['shot_statsbomb_xg'].mean() if n_shots > 0 else 0.0

    # Dribbling and Turnover Metrics
    res['dribble_rate'] = type_counts.get('Dribble', 0)
    res['turnover_rate'] = type_counts.get('Dispossessed', 0) + type_counts.get('Miscontrol', 0)

    # Defensive Metrics
    res['foul_rate'] = type_counts.get('Foul Committed', 0)
    res['press_intensity'] = type_counts.get('Pressure', 0)
    
    return res

In [3]:
def get_player_match_baseline_stats(player_events, target_player_id, subs, match_length_seconds, window_sec=300):
    if player_events.empty:
        return {}

    # End of match in seconds
    match_length_seconds = player_events['total_seconds'].max()
    
    # Determine the time window during which the player was on the pitch
    start_time = 0.0
    end_time = match_length_seconds
    sub_col = 'substitution_replacement_id'
    
    # Substitution check: if the player was substituted on, adjust start_time
    if sub_col in subs.columns:
        sub_on = subs[subs[sub_col] == target_player_id]
        if not sub_on.empty:
            start_time = sub_on['total_seconds'].values[0]
            
    # Substitution check: if the player was substituted off, adjust end_time
    sub_off = subs[subs['player_id'] == target_player_id]
    if not sub_off.empty:
        end_time = min(end_time, sub_off['total_seconds'].values[0])
        
    # Red card check: if the player received a red card, adjust end_time
    if 'foul_committed_card' in player_events.columns:
        red_foul = player_events[player_events['foul_committed_card'].isin(['Red Card', 'Second Yellow'])]
        if not red_foul.empty:
            end_time = min(end_time, red_foul['total_seconds'].values[0])
            
    if 'bad_behaviour_card' in player_events.columns:
        red_bb = player_events[player_events['bad_behaviour_card'].isin(['Red Card', 'Second Yellow'])]
        if not red_bb.empty:
            end_time = min(end_time, red_bb['total_seconds'].values[0])

    # Create time windows for the player's presence on the pitch
    match_windows_metrics = []
    
    valid_events = player_events[(player_events['total_seconds'] >= start_time) & (player_events['total_seconds'] < end_time)]
    
    if valid_events.empty:
        return {}
    
    for start_t in np.arange(start_time, end_time, window_sec):
        end_t = start_t + window_sec
        
        # If the end of the window exceeds the player's time on the pitch, break the loop (Only full 5 minute windows are considered)
        if end_t > end_time:
            break

        # Filter events for the current time window
        window_events = valid_events[(valid_events['total_seconds'] >= start_t) & (valid_events['total_seconds'] < end_t)]
        
        # Get player metrics for the current window
        metrics = get_player_metrics(window_events, target_player_id, window_sec=window_sec)
        match_windows_metrics.append(metrics)

    # Baseline statistics: mean and standard deviation for each metric across all windows
    if len(match_windows_metrics) < 2:
        # If there are fewer than 2 windows, we cannot compute standard deviation, so we return an empty dictionary
        return {}
        
    metrics_keys = match_windows_metrics[0].keys()
    baseline_stats = {}
    
    for k in metrics_keys:
        vals = [d[k] for d in match_windows_metrics]
        baseline_stats[f'mean_{k}'] = np.mean(vals)
        baseline_stats[f'std_{k}'] = np.std(vals, ddof=1)
        
    return baseline_stats

In [4]:
metrics_list = ['pass_acc_ratio', 'fwd_pass_ratio', 'long_ball_ratio', 'xg_rate', 'shot_rate', 'avg_xg', 'dribble_rate', 'turnover_rate', 'foul_rate', 'press_intensity']
final_aggregated_teammate_data = []
grouped_pde = df.groupby('match_id')

prots_dict = protagonists_df.groupby('event_id')['player_id'].apply(lambda x: x.dropna().tolist()).to_dict()

for m_id, match_events in tqdm(grouped_pde, desc="Processing Matches", unit="matches"):
    # Load match events from pickle file
    events = pd.read_pickle(f"sb_offline_data/match_{m_id}.pkl")
    events['total_seconds'] = events['minute'] * 60 + events['second']
    
    if 'location' in events.columns:
        locs = events['location'].dropna().tolist()
        if len(locs) > 0:
            events['x'] = events['location'].str[0]
            events['y'] = events['location'].str[1]
    
    if 'pass_end_location' in events.columns:
        events['end_x'] = events['pass_end_location'].str[0]
        events['end_y'] = events['pass_end_location'].str[1]
    else:
        events['end_x'] = np.nan
        events['end_y'] = np.nan
        
    match_length_seconds = events['total_seconds'].max() if not events.empty else 0
    subs = events[events['type'] == 'Substitution']
    
    player_events_dict = {k: v for k, v in events.groupby('player_id')}
        
    # Cache for match-level baseline statistics to avoid redundant calculations
    match_baselines_cache = {}

    for _, event_row in match_events.iterrows():
        t = event_row['total_seconds']
        team = event_row['affected_team']
        e_id = event_row['event_id']
        period = event_row['period']
        event_type = event_row.get('event_type', None)
        
        # Identify protagonists for the current event
        prots = prots_dict.get(e_id, [])
        
        # Identify active teammates within the 5-minute window around the event
        window_events = events[(events['total_seconds'] >= t - 300) & (events['total_seconds'] <= t + 300) & (events['team'] == team)]
        active_players = window_events['player_id'].dropna().unique()
        teammates = [p for p in active_players if p not in prots]
        
        # Get teammate metrics and compute z-scores
        event_teammate_z_scores = []
        for tm_id in teammates:
            player_events = player_events_dict.get(tm_id)
            if player_events is None or player_events.empty:
                continue
                
            # Calculate baseline statistics for the teammate if not already cached
            if tm_id not in match_baselines_cache:
                match_baselines_cache[tm_id] = get_player_match_baseline_stats(player_events, tm_id, subs, match_length_seconds, 300)
            baseline_stats = match_baselines_cache[tm_id]
            
            if not baseline_stats:
                continue
                
            # Pre/post event filtering for the 5-minute window for every teammate
            pre_events = player_events[(player_events['total_seconds'] >= t - 300) & (player_events['total_seconds'] < t) & (player_events['period'] == period)]
            post_events = player_events[(player_events['total_seconds'] > t) & (player_events['total_seconds'] <= t + 300) & (player_events['period'] == period)]
            
            # Get pre and post metrics for the teammate
            pre_m = get_player_metrics(pre_events, tm_id, event_type=event_type, is_post_window=False)
            post_m = get_player_metrics(post_events, tm_id, event_type=event_type, is_post_window=True)
            
            # Z-score calculation for each metric
            output = {}
            for metric in pre_m.keys():
                mean_val = baseline_stats.get(f'mean_{metric}')
                std_val = baseline_stats.get(f'std_{metric}')
                
                if std_val is not None and std_val > 0:
                    output[f'z_score_{metric}'] = (post_m[metric] - mean_val) / std_val
                elif post_m[metric] == mean_val:
                    output[f'z_score_{metric}'] = 0.0
                else:
                    output[f'z_score_{metric}'] = np.nan
                    
            event_teammate_z_scores.append(output)
            
        # Aggregate the z-scores for the teammates and compute the average for each metric
        agg_row = {'event_id': e_id}
        if event_teammate_z_scores:
            for m in metrics_list:
                z_col = f'z_score_{m}'
                valid_vals = [d[z_col] for d in event_teammate_z_scores if not np.isnan(d.get(z_col, np.nan))]
                agg_row[f'team_avg_z_{m}'] = np.mean(valid_vals) if valid_vals else np.nan
        else:
            for m in metrics_list:
                agg_row[f'team_avg_z_{m}'] = np.nan
                
        final_aggregated_teammate_data.append(agg_row)

# Final DataFrame containing the average z-scores for teammates for each event
teammates_avg_df = pd.DataFrame(final_aggregated_teammate_data)

Processing Matches: 100%|██████████| 1182/1182 [44:11<00:00,  2.24s/matches]


In [5]:
# Merge the protagonist data with the teammates' average z-scores
protagonists_df = pd.read_csv('output/individual/catalogue/protagonists_master.csv')
contagion_df = pd.merge(protagonists_df, teammates_avg_df, on='event_id', how='inner')

# Relative difference calculation for each metric
for m in metrics_list:
    prot_z_col = f'z_score_{m}'
    team_z_col = f'team_avg_z_{m}'
    rel_diff_col = f'relative_diff_z_{m}'
    
    if prot_z_col in contagion_df.columns and team_z_col in contagion_df.columns:
        contagion_df[rel_diff_col] = contagion_df[prot_z_col] - contagion_df[team_z_col]
        
# Save results to CSV if in PDE mode
if MODE == 'PDE':
    contagion_df.to_csv('output/individual/analysis/contagion_analysis_master.csv', index=False, encoding='utf-8-sig')